# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. These unique `@id`s are used in all subsequent operations.

In [ ]:
# List all available record sets and their field @ids
record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record set(s)\n")
for rs in record_sets:
    print(f"Record set name: {rs['name']}\n  @id: {rs['@id']}")
    print("  Fields:")
    for field in rs['field']:
        if isinstance(field, dict):
            print(f"    - {field['name']} (@id: {field['@id']})")
        else:
            # Occasionally, Croissant schemas can store fields as just @id instead of as a dict
            print(f"    - (@id: {field})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all available record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Record set IDs: {record_set_ids}\n")

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set '@id'={record_set_id}")
        print("Columns:", df.columns.tolist())
    except Exception as e:
        print(f"Could not load records from '{record_set_id}': {e}")
print()
# Display the first few rows of the main record set (choose the first as default)
main_record_set_id = record_set_ids[0] if len(record_set_ids) else None
if main_record_set_id is not None and main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes example operations: removing outliers, transforming distributions, and grouping.

In [ ]:
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to select a typical numeric field for demonstration
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_candidates:
        # Attempt to find columns with numeric-looking values
        for col in df.columns:
            # Try conversion for first value
            try:
                float(df[col].dropna().iloc[0])
                numeric_candidates.append(col)
            except:
                continue
    print(f"Numeric field candidates: {numeric_candidates}\n")
    numeric_field = numeric_candidates[0] if numeric_candidates else None
    
    if numeric_field:
        # Ensure numeric type
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanpercentile(df[numeric_field], 75)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (75th percentile): {len(filtered_df)} records\n")
        # Normalize the field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by another field if available (e.g., 'description', 'Sample', etc.)
        non_num_cols = [col for col in df.columns if col != numeric_field and df[col].nunique() < len(df) // 3 and df[col].dtype == object]
        group_field = non_num_cols[0] if non_num_cols else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped data by '{group_field}':")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    if numeric_field:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
        if group_field:
            plt.figure(figsize=(10, 5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.show()

## 6. Conclusion
This notebook demonstrated loading and initial exploration of the FAIR^2 mass spectrometry dataset using `mlcroissant`, including record set listing, DataFrame assembly, simple filtering, normalization, and visualization steps. For deeper analyses, refer to specific field definitions and documentation provided by the dataset authors.